# Prepare Calendar Dimension Table

Generates a standard calendar/date dimension table (`azuredatabricks_medical.silver.calendar`) spanning **1940-01-01** to **2030-12-31** with all common date attributes.

In [0]:
from pyspark.sql import functions as F

# Generate all dates from 1940-01-01 to 2030-12-31
start_date = "1940-01-01"
end_date = "2030-12-31"

date_df = spark.sql(f"SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS date")

# Build full calendar dimension
calendar_df = date_df.select(
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("date"),
    F.year("date").alias("year"),
    F.quarter("date").alias("quarter"),
    F.month("date").alias("month"),
    F.weekofyear("date").alias("week_of_year"),
    F.dayofmonth("date").alias("day_of_month"),
    F.dayofweek("date").alias("day_of_week"),
    F.dayofyear("date").alias("day_of_year"),
    F.date_format("date", "EEEE").alias("day_name"),
    F.date_format("date", "EEE").alias("day_name_short"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "MMM").alias("month_name_short"),
    F.concat(F.lit("Q"), F.quarter("date")).alias("quarter_name"),
    F.date_format("date", "yyyy-MM").alias("year_month"),
    F.concat(F.year("date"), F.lit("-Q"), F.quarter("date")).alias("year_quarter"),
    F.when(F.dayofweek("date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
    F.when(F.dayofweek("date").isin(1, 7), False).otherwise(True).alias("is_weekday"),
    F.when(
        (F.year("date") % 4 == 0) & ((F.year("date") % 100 != 0) | (F.year("date") % 400 == 0)),
        True
    ).otherwise(False).alias("is_leap_year"),
    F.trunc("date", "month").alias("first_day_of_month"),
    F.last_day("date").alias("last_day_of_month"),
    F.trunc("date", "quarter").alias("first_day_of_quarter"),
    F.date_sub(F.add_months(F.trunc("date", "quarter"), 3), 1).alias("last_day_of_quarter"),
    F.trunc("date", "year").alias("first_day_of_year"),
    F.make_date(F.year("date"), F.lit(12), F.lit(31)).alias("last_day_of_year"),
    F.datediff(F.col("date"), F.trunc("date", "month")).alias("day_of_month_index"),
    F.datediff(F.last_day("date"), F.col("date")).alias("days_remaining_in_month"),
    F.datediff(
        F.date_sub(F.add_months(F.trunc("date", "quarter"), 3), 1), F.col("date")
    ).alias("days_remaining_in_quarter")
)

print(f"Total calendar rows: {calendar_df.count()}")
calendar_df.printSchema()

In [0]:
target_table = "azuredatabricks_medical.silver.calendar"

calendar_df.write.mode("overwrite").saveAsTable(target_table)

print(f"Calendar table written to '{target_table}'")
display(spark.read.table(target_table).limit(10))